# Remote execution, in Jupyter

Run this notebook where you work (a laptop, or the access node of a cluster): every acquisition is sent to the machine with access to the instruments, through SSH and SLURM, while fit and report run here. No platform is created locally, and no instrument access is needed.

The machine running the acquisitions needs Qibocal (installed below) and the platform, found through its own `QIBOLAB_PLATFORMS`.

In [ ]:
import json
import os
import shutil
from pathlib import Path

from IPython.display import Markdown
from qibocal import protocols

from qibocal_widgets import Executor, Session, reports_dir
from qibocal_widgets.jupyter import Jupyter
from qibocal_widgets.remote import Connection, Remote, config_path, connections

HERE = Path.cwd()  # Jupyter runs notebooks in their folder

## 1. The connection

A connection is given inline, or by name. Named connections are stored in `remotes.json`, in the configuration shared with Qibocamp, whose desktop application can edit them too; `QIBOCAL_WIDGETS_CONNECTION` selects one without changing the notebook.

**Edit the inline connection** to match your cluster, or set `QIBOCAL_WIDGETS_CONNECTION` to the name of a stored one.

In [ ]:
name = os.environ.get("QIBOCAL_WIDGETS_CONNECTION")
if name is not None:
    remote = Remote(name)
else:
    remote = Remote(
        Connection(
            ssh="me@login.cluster",  # ssh arguments, ending with the destination
            queue="qpu",  # SLURM partition with the instruments
            slurm="--time=10",  # further srun options
            platform="qutrits",  # platform name, on the remote machine
            requirements="qibocal @ git+https://github.com/qiboteam/qibocal.git@protodev",
        )
    )
Markdown(
    f"`{remote!r}`\n\nStored connections, in `{config_path()}`: {sorted(connections())}"
)

To store the inline connection, e.g. as `lab`: `save_connection("lab", remote.connection)`; then use `Remote("lab")`.

## 2. The remote environment

Acquisitions run in a Python environment on the remote machine. Installing it is a separate step, never performed by the acquisitions: it is skipped when the requirements did not change, and runs on the access node, outside of the queue.

In [ ]:
remote.install()

## 3. The executor, without a local platform

The executor records every execution in its session folder, here, as local executions are. Within a Qibocamp session, the folder defaults to its reports folder (`QIBOCAMP_REPORTS_DIR`), so that calibrations show up in its dashboard.

In [ ]:
if reports_dir() is None:
    path = HERE / "var" / "remote"
    if path.exists():
        shutil.rmtree(path)
else:
    path = None

executor = Executor(
    None,
    path=path,
    targets=[0, 1],
    update=True,
    runner=remote,
    frontends=[Jupyter()],
)
executor.path

The platform state is asked to the remote machine, the first time it is needed: it can be explored, and edited, as a local one. Edits travel with the next acquisitions.

In [ ]:
start = executor.platform_state.snapshot()
executor.platform_state.explore("native_gates.single_qubit.0")

## 4. Acquiring remotely

The acquisition is a job in the queue (named `qibocal-widgets-acquire`); the data come back, are recorded in the session, and are fitted and reported here.

In [ ]:
with executor:
    ramsey = executor(protocols.ramsey(delay=(10, 1000, 100)), nshots=500)
executor.note("Acquired remotely", ramsey)
ramsey

## 5. Updating the remote platform

Without a local platform, updates run on the access node, outside of the queue; the runner keeps the new state, sends it with the next jobs, and the session records it (`new_platform/`).

In [ ]:
with executor:
    rabi = executor(
        protocols.rabi_amplitude_signal(
            min_amp=0.0, max_amp=0.6, step_amp=0.03, pulse_length=40
        ),
        nshots=500,
    )
rabi

In [ ]:
executor.platform_state.diff(start)

## 6. The queue

While acquisitions wait in the queue, it can be followed from here: the whole partition, with your jobs highlighted and the runner's tagged. The monitor refreshes periodically while displayed.

In [ ]:
remote.monitor(interval=30)

## 7. Continuing later

After restarting the kernel, a new executor attached to the session folder continues appending to it; the latest recorded platform state is handed to the runner, so that the calibration continues where it was left.

In [ ]:
later = Executor(
    None,
    session=Session.attach(executor.path),
    targets=[0, 1],
    update=False,
    runner=Remote(remote.connection),  # a fresh runner, as after a restart
    frontends=[Jupyter()],
)
{
    "history": json.loads((later.path / "history.json").read_text()),
    "restored": later.platform_state.diff(executor.platform_state) == {},
}